In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib
dataset_path = 'super_sms_dataset.csv'
sms_dataset = pd.read_csv(dataset_path, encoding='latin1')
model_path = 'spam_detection_svm_model.pkl'
vectorizer_path = 'tfidf_vectorizer.pkl'
svm_model = joblib.load(model_path)
tfidf_vectorizer = joblib.load(vectorizer_path)
def predict_spam(message):
    message_tfidf = tfidf_vectorizer.transform([message])
    prediction = svm_model.predict(message_tfidf)
    confidence = svm_model.predict_proba(message_tfidf).max() * 100
    if prediction[0] == 1:
        return f"The message is SPAM with {confidence:.2f}% confidence."
    else:
        return f"The message is NOT SPAM with {confidence:.2f}% confidence."

while True:
    user_input = input("Enter a text message (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        break
    print(predict_spam(user_input))
    
sms_dataset = sms_dataset.dropna()

sms_dataset.columns = ['Message', 'Label']

sms_dataset['Label'] = sms_dataset['Label'].astype(int)

X = sms_dataset['Message']
y = sms_dataset['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_tfidf, y_train)


y_pred = svm_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
joblib.dump(svm_model, 'spam_detection_svm_model.pkl')
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved successfully.")

Enter a text message (or type 'exit' to quit):  Hi, my name is Constantine. I will be helping you for the audit logging in IDM.


The message is NOT SPAM with 99.85% confidence.


Enter a text message (or type 'exit' to quit):  Hi Constantine, hope your are doing well. I have made changes in the audit.json and provisioner file as mentioned in the documentation, but the logs are still not reflecting the the DB.


The message is NOT SPAM with 99.98% confidence.


Enter a text message (or type 'exit' to quit):  Requesting your help


...

The message is SPAM with 99.58% confidence.


Enter a text message (or type 'exit' to quit):  I am a prince from Zandar and need your help transferring $10 million. Reply with your bank details.


The message is SPAM with 76.81% confidence.


Enter a text message (or type 'exit' to quit):  You are on fire. Play more games to earn upto rupees 10,00,000. Download Zupee now


The message is SPAM with 100.00% confidence.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

dataset_path = 'super_sms_dataset.csv'
sms_dataset = pd.read_csv(dataset_path, encoding='latin1')

sms_dataset = sms_dataset.dropna()

sms_dataset.columns = ['Message', 'Label']

sms_dataset['Label'] = sms_dataset['Label'].astype(int)

X = sms_dataset['Message']
y = sms_dataset['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_tfidf, y_train)

y_pred = svm_model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:\n", classification_report(y_test, y_pred))

conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

joblib.dump(svm_model, 'spam_detection_svm_model.pkl')
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved successfully.")


In [ ]:
import joblib

def load_model_and_vectorizer(model_path='spam_detection_svm_model.pkl', vectorizer_path='tfidf_vectorizer.pkl'):
    svm_model = joblib.load(model_path)
    tfidf_vectorizer = joblib.load(vectorizer_path)
    return svm_model, tfidf_vectorizer

def predict_spam(message, svm_model, tfidf_vectorizer):
    message_tfidf = tfidf_vectorizer.transform([message])
    prediction = svm_model.predict(message_tfidf)
    confidence = svm_model.predict_proba(message_tfidf).max() * 100
    return f"The message is {'SPAM' if prediction[0] == 1 else 'NOT SPAM'} with {confidence:.2f}% confidence."

if __name__ == "__main__":
    svm_model, tfidf_vectorizer = load_model_and_vectorizer()
    while True:
        user_input = input("Enter a text message (or type 'exit' to quit): ")
        if user_input.lower() == 'exit':
            break
        print(predict_spam(user_input, svm_model, tfidf_vectorizer))


Enter a text message (or type 'exit' to quit):  Subject: URGENT: Your Account Has Been Selected for a Special Reward!  Dear Valued Customer,  Congratulations! You have been specially selected in our Exclusive Customer Loyalty Program to receive a cash prize of $500,000 USD! This is part of our global initiative to reward our loyal customers and email subscribers.  Our records indicate that your email address was randomly selected from our database of verified users, and after a thorough verification process, we are pleased to inform you that you qualify for this incredible once-in-a-lifetime opportunity!  To claim your prize, all you need to do is verify your identity and confirm your details by providing the following information:  Full Name:  Address:  Phone Number:  Email Address:  Bank Account Details (For Secure Funds Transfer):  Please respond within 24 hours to avoid forfeiting this reward. Failure to reply in time will result in the prize being transferred to the next eligible 

The message is SPAM with 100.00% confidence.


Enter a text message (or type 'exit' to quit):  I hope you're doing well. I wanted to take a moment to share some exciting insights into the evolving world of Artificial Intelligence (AI) and Cybersecurity—two fields that are rapidly converging to shape the future of digital security.  As you may know, cyber threats have become increasingly sophisticated, with attackers leveraging AI to automate phishing, social engineering, and malware distribution. However, the good news is that AI-driven security solutions are also evolving at a remarkable pace, helping organizations detect and mitigate threats in real time.


The message is SPAM with 87.13% confidence.


Enter a text message (or type 'exit' to quit):  Hi, Constantine. I am facing issues while trying to configure the IDM for audit logging using JDBC for my Oracle DB. I have made the changes in the tools directory of the project and edited the files mentioned in the documentation on backstage platform. The issue still persists and the logs are not reflecting in the DB. Requesting your help. Thanks & Regards, Armaan.


The message is NOT SPAM with 96.98% confidence.
